# Session 3 — Normative vs Participant Ratings
**Participant:** CPEEG02 | **Session:** 3 | **Date:** 2026-06-17

In [ ]:
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

In [ ]:
# ── Load normative ratings ──────────────────────────────────────────────────
csv_path = 'cpeeg02_session_3.csv'
norm_df = pd.read_csv(csv_path)
norm_df.columns = norm_df.columns.str.strip()
print(norm_df.head())
print(f'\n{len(norm_df)} trials, columns: {list(norm_df.columns)}')

In [ ]:
# ── Load participant ratings from MAT (v7.3 / HDF5) ──────────────────────────
mat_path = 'Results_CPEEG02_S3_C3_B2_2026-06-17.mat'

def read_ref_array(f, dataset):
    results = []
    for ref in dataset[:, 0]:
        obj = f[ref]
        val = obj[()]
        if val.dtype.kind in ('u', 'i'):
            results.append(''.join(chr(c) for c in val.flatten()))
        else:
            results.append(float(val.flatten()[0]) if val.size == 1 else val.flatten())
    return results

with h5py.File(mat_path, 'r') as f:
    trials = f['Results/Trials']
    trial_idx   = read_ref_array(f, trials['ImageIndex'])
    part_rating = read_ref_array(f, trials['Valence'])
    condition   = read_ref_array(f, trials['Condition'])
    rt          = read_ref_array(f, trials['RT_Valence'])

part_df = pd.DataFrame({
    'trial':       [int(x) for x in trial_idx],
    'part_rating': part_rating,
    'condition':   condition,
    'rt':          rt
})
print(part_df.head())

In [ ]:
# ── Merge on trial number ─────────────────────────────────────────────────────
df = pd.merge(norm_df, part_df, on='trial')
df = df.sort_values('trial').reset_index(drop=True)
# Both CSV and MAT have a 'condition' column → pandas appends _x/_y; keep MAT version
if 'condition_x' in df.columns:
    df = df.rename(columns={'condition_y': 'condition'}).drop(columns=['condition_x'])
print(f'Merged: {len(df)} trials')
print(df[['trial','Category','Valence','part_rating','condition']].head(10))

In [ ]:
# ── Figure 1: Trial-by-trial comparison ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df['trial'], df['Valence'],     'o-', lw=1.2, ms=4, label='Normative', color='steelblue', alpha=0.8)
ax.plot(df['trial'], df['part_rating'], 's-', lw=1.2, ms=4, label='Participant', color='tomato', alpha=0.8)
ax.set_xlabel('Trial')
ax.set_ylabel('Valence Rating (1–9)')
ax.set_title('Session 3 — Normative vs Participant Ratings (by trial order)')
ax.legend()
ax.set_ylim(0.5, 9.5)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('s3_trial_timecourse.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Figure 2: Scatter — normative vs participant ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = {'FEEL': '#E87040', 'TONE': '#4C72B0'}

for ax, cond in zip(axes, ['FEEL', 'TONE']):
    sub = df[df['condition'] == cond]
    r, p = stats.pearsonr(sub['Valence'], sub['part_rating'])
    ax.scatter(sub['Valence'], sub['part_rating'],
               c=colors[cond], alpha=0.7, edgecolors='k', lw=0.4, s=60)
    m, b = np.polyfit(sub['Valence'], sub['part_rating'], 1)
    x_line = np.linspace(1, 9, 100)
    ax.plot(x_line, m*x_line + b, '--', color='gray', lw=1.5)
    ax.plot([1,9],[1,9], 'k:', lw=1, alpha=0.4, label='identity')
    ax.set_xlabel('Normative Valence')
    ax.set_ylabel('Participant Rating')
    ax.set_title(f'Session 3 — {cond}  (r={r:.2f}, p={p:.3f})')
    ax.set_xlim(0.5, 9.5); ax.set_ylim(0.5, 9.5)
    ax.set_aspect('equal')
    ax.grid(alpha=0.3)

plt.suptitle('Session 3 — Scatter: Normative vs Participant', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('s3_scatter.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Figure 3: By category ─────────────────────────────────────────────────────
categories = sorted(df['Category'].unique())
x = np.arange(len(categories))
width = 0.35

norm_means  = [df[df['Category']==c]['Valence'].mean()      for c in categories]
norm_sems   = [df[df['Category']==c]['Valence'].sem()       for c in categories]
part_means  = [df[df['Category']==c]['part_rating'].mean()  for c in categories]
part_sems   = [df[df['Category']==c]['part_rating'].sem()   for c in categories]

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - width/2, norm_means, width, yerr=norm_sems, capsize=4,
       label='Normative', color='steelblue', alpha=0.85, error_kw={'elinewidth':1.5})
ax.bar(x + width/2, part_means, width, yerr=part_sems, capsize=4,
       label='Participant', color='tomato', alpha=0.85, error_kw={'elinewidth':1.5})
ax.set_xticks(x); ax.set_xticklabels(categories)
ax.set_ylabel('Mean Valence Rating (±SEM)')
ax.set_title('Session 3 — Mean Ratings by Image Category')
ax.set_ylim(0, 10)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('s3_by_category.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Figure 4: FEEL vs TONE — normative vs participant ────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
conditions = ['FEEL', 'TONE']
x = np.arange(len(conditions))
width = 0.35

n_means = [df[df['condition']==c]['Valence'].mean()      for c in conditions]
n_sems  = [df[df['condition']==c]['Valence'].sem()       for c in conditions]
p_means = [df[df['condition']==c]['part_rating'].mean()  for c in conditions]
p_sems  = [df[df['condition']==c]['part_rating'].sem()   for c in conditions]

ax.bar(x - width/2, n_means, width, yerr=n_sems, capsize=5,
       label='Normative', color='steelblue', alpha=0.85)
ax.bar(x + width/2, p_means, width, yerr=p_sems, capsize=5,
       label='Participant', color='tomato', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(conditions)
ax.set_ylabel('Mean Valence Rating (±SEM)')
ax.set_title('Session 3 — Mean Ratings by Condition')
ax.set_ylim(0, 10)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('s3_by_condition.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Summary stats ─────────────────────────────────────────────────────────────
r_all, p_all = stats.pearsonr(df['Valence'], df['part_rating'])
print(f'=== Session 3 Summary ===')
print(f'Overall Pearson r = {r_all:.3f}  (p = {p_all:.4f})')
print(f'\nNormative:   mean={df["Valence"].mean():.2f}, std={df["Valence"].std():.2f}')
print(f'Participant: mean={df["part_rating"].mean():.2f}, std={df["part_rating"].std():.2f}')
print(f'\nMean RT: {df["rt"].mean():.2f}s  (range {df["rt"].min():.2f}–{df["rt"].max():.2f}s)')
for cond in ['FEEL','TONE']:
    sub = df[df['condition']==cond]
    r, p = stats.pearsonr(sub['Valence'], sub['part_rating'])
    print(f'  {cond}: r={r:.3f}, p={p:.4f}, n={len(sub)}')